# Task 4: Testing Whether Hammer Weights Have Become Less Consistent

## Given info

The assigned branch is $\xi_{13}=3$, which asks whether the new production system produces less consistent weights.

Less consistent weights are interpreted as greater population variance.

The historical production data summarize 1,000 hammers:

$$
n_O=1000,\qquad
\overline{x}_O=993\ \text{g},\qquad
s_O=34.9\ \text{g}.
$$

The new-system sample contains the following weights, in grams:

$$
1018,\ 855,\ 1161,\ 1157,\ 916,\ 1079,\ 1060,\ 1005,\ 1013,\ 1154.
$$

## Objectives

1. Define a probability model and its assumptions
2. Formulate hypotheses about the population variances
3. Select and justify a statistical test
4. State a decision rule using a critical value
5. Calculate the test statistic and p-value
6. Explain Type I and Type II errors and calculate relevant probabilities
7. Interpret the result in the production context

## Probability model and assumptions

Let $O_i$ denote the weight of a hammer produced by the old system and $N_j$ the weight of a hammer produced by the new system.

The working models are

$$
O_i\sim\mathcal{N}(\mu_O,\sigma_O^2)
$$

and

$$
N_j\sim\mathcal{N}(\mu_N,\sigma_N^2).
$$

The model parameters are the two population means and variances:

$$
\mu_O,\quad \sigma_O^2,\quad \mu_N,\quad \sigma_N^2.
$$

The reported historical mean and standard deviation are interpreted as sample estimates from the 1,000 historical hammers. In particular, 34.9 g is not treated as a perfectly known population standard deviation.

The following assumptions are used:

1. Observations are independent within each production sample
2. The historical and new samples are independent of one another
3. Each sample represents a stable production process with a common mean and variance during its observation period
4. The weights in each population follow a normal distribution
5. The reported historical standard deviation uses the usual sample variance denominator, $n_O-1$
6. The measurements are comparable, and differences in measurement error or calibration do not explain the observed variability

Normality is a working approximation for variation around a manufacturing target. It is not established by the information supplied.

The historical raw observations are unavailable, and the new sample contains only ten observations. These limitations prevent strong empirical verification of the assumptions.

The normal model also permits negative weights mathematically, although such values are physically impossible. Its use is an approximation appropriate only when that impossible region has negligible probability.

The means are allowed to differ. The question addressed here concerns variability rather than a change in average weight.

## Hypotheses and significance level

Let

$$
\rho=\frac{\sigma_N^2}{\sigma_O^2}
$$

denote the ratio of the new-system variance to the old-system variance.

The hypotheses are

$$
H_0:\rho\leq1
$$

against

$$
H_1:\rho>1.
$$

Thus, the alternative states that the new production system has greater weight variability.

An upper-tailed test is appropriate because the question specifies an increase in variance.

The significance level is set to

$$
\alpha=0.05.
$$

This is a conventional working choice in the absence of a specified cost-based threshold. Under the model, it limits the probability of incorrectly concluding that variability increased to at most 5% when the null hypothesis holds.

The rejection threshold is calibrated at the null boundary $\rho=1$.

In [2]:
import numpy as np
from scipy.stats import f

# Historical sample summary.
n_old = 1000
old_mean = 993.0
old_sd = 34.9
old_variance = old_sd**2

# New-system weights, in grams.
new_weights = np.array([
    1018, 855, 1161, 1157, 916,
    1079, 1060, 1005, 1013, 1154
], dtype=float)

alpha = 0.05

n_new = len(new_weights)
new_mean = float(np.mean(new_weights))

# ddof=1 uses the sample variance denominator n-1.
new_variance = float(np.var(new_weights, ddof=1))
new_sd = float(np.sqrt(new_variance))

assert n_old > 1
assert n_new > 1
assert old_variance > 0
assert new_variance > 0

print(f"Historical sample size: {n_old}")
print(f"Historical sample variance: {old_variance:.2f} g^2")
print(f"New sample size: {n_new}")
print(f"New sample mean: {new_mean:.1f} g")
print(f"New sample variance: {new_variance:.6f} g^2")
print(f"New sample standard deviation: {new_sd:.6f} g")

Historical sample size: 1000
Historical sample variance: 1218.01 g^2
New sample size: 10
New sample mean: 1041.8 g
New sample variance: 10583.733333 g^2
New sample standard deviation: 102.877273 g


## Test statistic and reference distribution

The new sample variance is

$$
s_N^2
=
\frac{1}{n_N-1}
\sum_{j=1}^{n_N}(N_j-\overline{N})^2.
$$

For independent normal samples, the standardized sample variances satisfy

$$
\frac{(n_N-1)S_N^2}{\sigma_N^2}
\sim\chi^2_{n_N-1}
$$

and

$$
\frac{(n_O-1)S_O^2}{\sigma_O^2}
\sim\chi^2_{n_O-1},
$$

independently.

Taking their ratio after division by their degrees of freedom gives

$$
\frac{S_N^2/\sigma_N^2}{S_O^2/\sigma_O^2}
=
\frac{S_N^2/S_O^2}{\rho}
\sim F_{n_N-1,n_O-1}.
$$

At the null boundary $\rho=1$, the test statistic

$$
F_{\mathrm{obs}}=\frac{s_N^2}{s_O^2}
$$

is compared with an F distribution having

$$
\nu_1=n_N-1=9,\qquad
\nu_2=n_O-1=999
$$

degrees of freedom.

The new-system variance is placed in the numerator because large values of the ratio provide evidence for increased variability.

## Decision rule

Let $G$ denote the CDF of the $F_{9,999}$ distribution, and define

$$
c=G^{-1}(1-\alpha)=G^{-1}(0.95).
$$

This notation defines c as the lower-tail 95th percentile.

The decision rule is

$$
F_{\mathrm{obs}}>c
\quad\Longrightarrow\quad
\text{reject }H_0,
$$

and

$$
F_{\mathrm{obs}}\leq c
\quad\Longrightarrow\quad
\text{do not reject }H_0.
$$

The one-sided p-value is

$$
p_{\mathrm{value}}=1-G(F_{\mathrm{obs}}).
$$

Equivalently, reject the null hypothesis when
$$
p_{\mathrm{value}}<\alpha
$$

Numerically, the upper-tail probability is evaluated directly using the survival function to avoid loss of precision when the CDF is very close to one.

In [3]:
df_numerator = n_new - 1
df_denominator = n_old - 1

f_observed = new_variance / old_variance

critical_value = float(
    f.ppf(1 - alpha, df_numerator, df_denominator)
)

p_value = float(
    f.sf(f_observed, df_numerator, df_denominator)
)

reject_by_critical_value = f_observed > critical_value
reject_by_p_value = p_value < alpha

# equivalent methods should agree
assert reject_by_critical_value == reject_by_p_value

# prob above crit value
assert np.isclose(
    f.sf(critical_value, df_numerator, df_denominator),
    alpha,
    rtol=1e-10,
    atol=1e-12,
)

print(f"Numerator degrees of freedom: {df_numerator}")
print(f"Denominator degrees of freedom: {df_denominator}")
print(f"Observed F statistic: {f_observed:.6f}")
print(f"Critical value: {critical_value:.6f}")
print(f"One-sided p-value: {p_value:.6e}")
print(f"Reject H0: {reject_by_critical_value}")

Numerator degrees of freedom: 9
Denominator degrees of freedom: 999
Observed F statistic: 8.689365
Critical value: 1.889236
One-sided p-value: 1.227172e-12
Reject H0: True


## Test result

The observed sample variance ratio is

$$
F_{\mathrm{obs}}
=
\frac{10583.733333}{1218.01}
\approx8.689365.
$$

The upper critical value at significance level $\alpha=0.05$ is

$$
c\approx1.889236.
$$

Since

$$
8.689365>1.889236,
$$

the null hypothesis is rejected.

The one-sided p-value is approximately

$$
p_{\mathrm{value}}
=
1.227172\times10^{-12},
$$

which is also below 0.05.

Under the stated model, the observations provide strong evidence that the new production system has greater weight variance.

The p-value is a tail probability calculated under the null boundary. It is not the probability that the null hypothesis is true.

## Error probabilities and power

### Type I error

A Type I error occurs if the test concludes that the new system has greater variance when its variance is actually no greater than that of the old system.

For a true variance ratio $\rho$,

$$
\frac{F_{\mathrm{obs}}}{\rho}\sim F_{9,999}.
$$

Consequently, the rejection probability is

$$
P_\rho(F_{\mathrm{obs}}>c)
=
1-G\left(\frac{c}{\rho}\right).
$$

At the null boundary $\rho=1$, this probability is $\alpha=0.05$.
For $\rho<1$, it is smaller than 0.05.

Thus, the test has a maximum Type I error probability of 5% under the stated assumptions.

### Type II error

A Type II error occurs if the test fails to detect an actual increase in population variance.

For a specified alternative variance ratio $\rho>1$,

$$
\beta(\rho)
=
P_\rho(F_{\mathrm{obs}}\leq c)
=
G\left(\frac{c}{\rho}\right).
$$

The power is

$$
1-\beta(\rho)
=
1-G\left(\frac{c}{\rho}\right).
$$

There is no single Type II error probability without specifying the size of the true increase.

The following calculations examine illustrative true variance ratios of 1.5, 2, and 4. These are hypothetical effect sizes, not additional observations or estimates of the true ratio.

In [4]:
type_i_at_boundary = float(
    f.sf(critical_value, df_numerator, df_denominator)
)

print(
    f"Type I error at equal population variances: "
    f"{type_i_at_boundary:.4f}"
)
print()
print("True variance ratio | Type II error | Power")

power_results = []

for true_ratio in (1.5, 2.0, 4.0):
    beta = float(
        f.cdf(
            critical_value / true_ratio,
            df_numerator,
            df_denominator,
        )
    )

    power = float(
        f.sf(
            critical_value / true_ratio,
            df_numerator,
            df_denominator,
        )
    )

    assert np.isclose(beta + power, 1.0)

    power_results.append((true_ratio, beta, power))

    print(
        f"{true_ratio:19.1f} | "
        f"{beta:13.6f} | "
        f"{power:.6f}"
    )

Type I error at equal population variances: 0.0500

True variance ratio | Type II error | Power
                1.5 |      0.745008 | 0.254992
                2.0 |      0.514888 | 0.485112
                4.0 |      0.106274 | 0.893726


### Interpretation of error probabilities

At equal population variances, the test incorrectly rejects the null hypothesis in 5% of repeated independent sampling experiments under the model.

If the new population variance were twice the old variance, the probability of failing to detect that increase would be approximately 51.49%, giving power of approximately 48.51%.

If the new population variance were four times the old variance, power would increase to approximately 89.37%.

These probabilities describe the test's repeated-sampling performance at specified true variance ratios. They are not probabilities that the decision made for the observed sample is correct or incorrect.